<a href="https://colab.research.google.com/github/DharaCS23181/Skincare_Product_Prediction/blob/main/RIDGE__CLASSIFIER(70_30).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [48]:
import pandas as pd
import tensorflow as tf

from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [49]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/ML/skin_recommendation_dataset.csv")

In [50]:

import pandas as pd
import numpy as np
import re

from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score

# ==============================
# 1. CLEAN FUNCTION
# ==============================

def clean_text(x):
    x = str(x)
    x = re.sub(r"[^a-zA-Z0-9,\-\s]", "", x)
    return [i.strip().lower() for i in x.split(",") if i.strip() != ""]


# ==============================


data = df[['skintype','skin_condition','notable_effects','product_type']].copy()

# clean columns
data['skintype'] = data['skintype'].apply(clean_text)
data['skin_condition'] = data['skin_condition'].apply(clean_text)
data['notable_effects'] = data['notable_effects'].apply(clean_text)


# ==============================
# 3. ENCODING
# ==============================

mlb_skin = MultiLabelBinarizer()
mlb_condition = MultiLabelBinarizer()
mlb_effects = MultiLabelBinarizer()

skin_features = pd.DataFrame(
    mlb_skin.fit_transform(data['skintype']),
    columns=mlb_skin.classes_
)

condition_features = pd.DataFrame(
    mlb_condition.fit_transform(data['skin_condition']),
    columns=mlb_condition.classes_
)

effects_features = pd.DataFrame(
    mlb_effects.fit_transform(data['notable_effects']),
    columns=mlb_effects.classes_
)

X = pd.concat([skin_features, condition_features, effects_features], axis=1)

le = LabelEncoder()
y = le.fit_transform(data['product_type'])


# ==============================
# 4. TRAIN TEST SPLIT (70-30)
# ==============================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=42
)


# ==============================
# 5. SCALING (IMPORTANT FOR RIDGE)
# ==============================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# ==============================
# 6. TRAIN RIDGE MODEL
# ==============================

ridge = RidgeClassifier(alpha=1.0)

ridge.fit(X_train_scaled, y_train)


# ==============================
# 7. EVALUATION
# ==============================

y_pred = ridge.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Weighted F1:", f1_score(y_test, y_pred, average='weighted'))
print(classification_report(y_test, y_pred))


# ==============================
# 8. USER INPUT (NUMBER BASED)
# ==============================

print("\nSelect Skin Type:")
for i, val in enumerate(mlb_skin.classes_):
    print(i, ":", val)

skin_choice = int(input("Enter number: "))
user_skin = [mlb_skin.classes_[skin_choice]]


print("\nSelect Skin Condition:")
for i, val in enumerate(mlb_condition.classes_):
    print(i, ":", val)

cond_choice = int(input("Enter number: "))
user_condition = [mlb_condition.classes_[cond_choice]]


print("\nSelect Effect:")
for i, val in enumerate(mlb_effects.classes_):
    print(i, ":", val)

effect_choice = int(input("Enter number: "))
user_effect = [mlb_effects.classes_[effect_choice]]


# ==============================
# 9. CREATE INPUT
# ==============================

user_df = pd.DataFrame(columns=X.columns)
user_df.loc[0] = 0

for val in user_skin + user_condition + user_effect:
    if val in user_df.columns:
        user_df.loc[0, val] = 1


# scale input
user_input_scaled = scaler.transform(user_df)


# ==============================
# 10. PREDICT (RIDGE)
# ==============================

prediction = ridge.predict(user_input_scaled)
predicted_type = le.inverse_transform(prediction)

print("\n✅ Predicted Product Type:", predicted_type[0])


# ==============================
# 11. SMART RECOMMENDATION SYSTEM
# ==============================

filtered = df[df['product_type'] == predicted_type[0]].copy()

filtered['notable_effects'] = filtered['notable_effects'].astype(str).str.lower()
filtered['skin_condition'] = filtered['skin_condition'].astype(str).str.lower()
filtered['skintype'] = filtered['skintype'].astype(str).str.lower()


def calculate_score(row):
    score = 0

    if user_effect[0] in row['notable_effects']:
        score += 3

    if user_condition[0] in row['skin_condition']:
        score += 2

    if user_skin[0] in row['skintype']:
        score += 1

    return score


filtered['score'] = filtered.apply(calculate_score, axis=1)
filtered = filtered.sort_values(by='score', ascending=False)


# ==============================
# 12. TOP 3 RESULTS
# ==============================

print("\n🔥 Top Recommended Products:\n")

top_results = filtered.head(3)

for i, row in top_results.iterrows():
    print("🔹 Product Name:", row['product_name'])
    print("🔹 Brand:", row['brand'])
    print("🔹 Effects:", ", ".join(clean_text(row['notable_effects'])))
    print("🔹 Score:", row['score'])
    print("🔹 Image URL:", row['picture_src'])
    print("-"*40)

Accuracy: 0.4673913043478261
Weighted F1: 0.45888467776563624
              precision    recall  f1-score   support

           0       0.46      0.46      0.46        61
           1       0.33      0.24      0.28        74
           2       0.46      0.66      0.54        92
           3       0.78      0.67      0.72        64
           4       0.34      0.29      0.31        77

    accuracy                           0.47       368
   macro avg       0.47      0.46      0.46       368
weighted avg       0.46      0.47      0.46       368


Select Skin Type:
0 : combination
1 : dry
2 : normal
3 : oily
4 : sensitive
Enter number: 1

Select Skin Condition:
0 : acne
1 : dry and dehydrated skin
2 : dull skin
3 : enlarged pores
4 : impaired skin barrier
5 : oily skin
6 : pigmentation
7 : redness
8 : skin imbalance
9 : sun damage
10 : sun protection
11 : wrinkles
Enter number: 4

Select Effect:
0 : acne-free
1 : acne-spot
2 : anti-aging
3 : balancing
4 : black-spot
5 : brightening
6 : h